# T-332R — Reranker **retrain** with mixed negatives (3 hard : 1 random)

**ResearchGPT**, Phase 3C. This is a *re-run* of notebook `02`, changing exactly
one thing: **what the negatives are made of.**

| | |
|---|---|
| Task | T-332R (retrain) → T-333 (export) → T-334/T-335 (adoption gate) |
| Base | `cross-encoder/ms-marco-MiniLM-L-6-v2` (~22M) — unchanged |
| Data | The **same** `train_triples.jsonl` you already uploaded — **no new upload** |
| Change | negatives go from **4 hard** to **3 hard + 1 random** per positive |
| Runtime | GPU T4 x2 (pinned to one device); **~30–45 min** end to end |
| Internet | **must be ON** |

---

## Why this run exists

The first fine-tune (2026-08-27) **won on every ranking metric and was rejected
anyway.** Out-of-domain Recall@20 rose 0.4815 → 0.6481 (+35% relative), in-domain
nDCG@10 rose 0.2646 → 0.7500 — and then the first real pipeline run (M1) showed
this, on the query `multi agent retrieval literature review`:

| rank | stock | tuned |
|---|---|---|
| 1 | LiRA (multi-agent literature review) **+7.77** | **IoT behavioural monitoring +4.77** |
| 2 | LatteReview (systematic review) +5.43 | LiRA, demoted to **+2.10** |

**The tuned model ranked an IoT-monitoring paper above the single most relevant
paper in the pool.** It scored wind-turbine fatigue, soil nitrogen and opera
libretti at **+3…+5** where stock gives **≈ −11**. It had not learned to rank
better; it had *stopped being able to recognise irrelevance*.

### The cause, and the fix

`T-331` mined **100% BM25 hard negatives**. Every negative was a topically
adjacent scientific abstract from SciReviewGen's own cited-abstract pool, so the
only question the model was ever asked was:

> *"which of these several plausible science abstracts belongs to this chapter?"*

It was **never** asked *"is this relevant at all?"* — so it stopped answering
that question. T-331's warning that *"random negatives make the task trivial"* is
right about **necessity** and wrong at **100%**. The fix is a **mixture**, not a
switch.

This notebook keeps the pair count, the queries and the 1:4 positive:negative
ratio **identical**, and changes only the composition of the negatives.

---

## What is different from notebook 02

1. **Section 4 is new** — it rebuilds the negatives as 3 hard : 1 random,
   in-notebook, from the file you already uploaded.
2. **Sections 7 and 9 run a relevance control**, the diagnostic that caught the
   broken model *after* it had already been adopted. It now runs **here**, before
   you download 91 MB and re-run the local gate.
3. **Section 10 is a go/no-go gate** with a threshold fixed *before* the run.

### Read before running
* ⚠️ **Section 1 pins `CUDA_VISIBLE_DEVICES=0`, and that line is load-bearing.**
  With two GPUs visible, `sentence-transformers` wraps the model in
  `DataParallel`, whose `preprocess` attribute the loss cannot reach — training
  dies with `AttributeError: 'DataParallel' object has no attribute 'preprocess'`.
  Measured on Kaggle 2026-08-27. On recurrence, **restart the kernel**; the pin
  is useless once CUDA has initialised.
* **Attach the same Kaggle Dataset as before.** This notebook reads the identical
  `train_triples.jsonl` and does the mixing itself.
* **The adoption gate is still out-of-domain and still local** (T-335). This
  notebook cannot run it. Section 10's gate is a *pre-filter*, not the decision.

## 1 · Configuration

Training hyper-parameters are `TRAINING_SPEC §2.3` verbatim and **unchanged from
notebook 02** — the point of this run is to isolate the effect of the negatives,
so nothing else may move.

`SEED` still equals `researchgpt.rerank.triples.SEED`, because the uploaded
triples were built with it. The negative *mixing* uses its own `MIX_SEED`, so the
survey split stays byte-identical to notebook 02's and the two runs remain
comparable.

In [ ]:
# ---------------------------------------------------------------------------
# PIN TO ONE GPU. Must run BEFORE torch initialises CUDA, i.e. before any import
# of torch or any call to torch.cuda.* -- hence the very first statement here.
#
# Kaggle's "GPU T4 x2" exposes two devices. sentence-transformers then wraps the
# model in torch.nn.DataParallel, and BinaryCrossEntropyLoss calls
# `self.model.preprocess(...)`, which DataParallel does not forward. Training
# dies mid-run with:
#     AttributeError: 'DataParallel' object has no attribute 'preprocess'
# Measured on Kaggle 2026-08-27. A 22M cross-encoder gains nothing from a second
# GPU anyway -- DataParallel would mostly add transfer overhead.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# ---------------------------------------------------------------------------

BASE_MODEL   = "cross-encoder/ms-marco-MiniLM-L-6-v2"   # ~22M, already in the pipeline

SEED         = 20260827      # must equal researchgpt.rerank.triples.SEED
EPOCHS       = 2             # TRAINING_SPEC 2.3. The 10-epoch build moved nDCG by -0.001.
LEARNING_RATE = 2e-5
BATCH_SIZE   = 32
MAX_LENGTH   = 512           # truncate the ABSTRACT, never the query
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
DEV_FRAC     = 0.10          # held out BY SURVEY, not by row

# --- THE ONLY REAL CHANGE IN THIS NOTEBOOK ---------------------------------
# 4 negatives per positive, as before, but no longer all hard.
#
#   notebook 02:  4 hard + 0 random   -> model forgot what irrelevance is
#   notebook 03:  3 hard + 1 random   -> keeps the discriminative signal AND
#                                        restores the relevance prior
#
# The hard negatives are the only signal a cross-encoder can add over BM25, so
# they stay in the majority. The random quarter is what teaches "not relevant at
# all", which is the property M1 proved was missing.
HARD_PER_POSITIVE   = 3
RANDOM_PER_POSITIVE = 1
MIX_SEED            = 20260828   # separate: keeps the survey split identical to nb 02
# ---------------------------------------------------------------------------

# Fixed BEFORE the run, so the verdict cannot be rationalised after seeing it.
# (Same discipline as decisions.txt D-09, which fixed the verifier's 0.75 gate
# before notebook 01 was ever executed.)
MAX_CONTROL_VIOLATION_RATE = 0.20   # tuned model must still recognise irrelevance
MIN_NDCG_GAIN              = 0.001  # and must still actually rank better

OUTPUT_DIR   = "/kaggle/working/reranker-mixed-negatives"

# Where the uploaded triples land. SAME FILE as notebook 02 -- no new upload.
TRIPLES_GLOB = "/kaggle/input/**/train_triples.jsonl"

print("base model :", BASE_MODEL)
print("negatives  :", f"{HARD_PER_POSITIVE} hard : {RANDOM_PER_POSITIVE} random per positive")
print("gate       :", f"control violations <= {MAX_CONTROL_VIOLATION_RATE:.0%}, "
      f"nDCG@10 gain >= {MIN_NDCG_GAIN}")

## 2 · Dependencies

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, mod=None):
    try:
        importlib.import_module(mod or pkg.replace("-", "_"))
        print(f"  present : {pkg}")
    except ImportError:
        print(f"  install : {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, mod in [("transformers", "transformers"),
                 ("accelerate>=0.26", "accelerate"),
                 ("sentence-transformers", "sentence_transformers"),
                 ("scikit-learn", "sklearn")]:
    ensure(pkg, mod)

import torch
print("\ntorch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("visible GPUs:", torch.cuda.device_count(), "(section 1 pins this to 1)")
assert torch.cuda.device_count() <= 1, (
    "More than one GPU is visible. sentence-transformers will wrap the model "
    "in DataParallel, and training dies with: AttributeError: DataParallel "
    "object has no attribute preprocess. Restart the kernel and Run All - "
    "CUDA_VISIBLE_DEVICES must be set in section 1 BEFORE torch "
    "initialises CUDA."
)

## 3 · Load the triples and recover the document pool

The uploaded file carries, per row, `query` / `text` / `label` / `doc_id`.

**Every one of the 13,929 cited abstracts appears in it** — verified locally
before this notebook was written: each document is a positive for at least one
chapter, because the corpus *is* the set of cited abstracts. That is what makes
section 4 possible without a second upload.

The cell below rebuilds three things:

* `doc_text` — the full `doc_id → abstract` corpus, to sample random negatives from
* `positives[query]` — the true positives, so a random draw can never hit one
* `hard[query]` — the mined BM25 negatives **in rank order**, so keeping the
  first 3 per positive keeps the *hardest* ones rather than an arbitrary subset

In [ ]:
import glob, json, random
from collections import Counter, OrderedDict

paths = glob.glob(TRIPLES_GLOB, recursive=True)
assert paths, (
    f"No triples found at {TRIPLES_GLOB}.\n"
    "Attach the SAME Kaggle Dataset you used for notebook 02 "
    "(Add Input -> Datasets). See notebook-kaggle-traning/README.md."
)
print("triples file:", paths[0])

doc_text   = {}                 # doc_id -> abstract
positives  = OrderedDict()      # query  -> [doc_id]   (true positives)
hard       = OrderedDict()      # query  -> [doc_id]   (BM25 rank order, preserved)

n_rows = 0
for line in open(paths[0], encoding="utf-8"):
    if not line.strip():
        continue
    r = json.loads(line)
    n_rows += 1
    doc_text.setdefault(r["doc_id"], r["text"])
    if r["label"] == 1:
        positives.setdefault(r["query"], []).append(r["doc_id"])
        hard.setdefault(r["query"], [])
    else:
        hard.setdefault(r["query"], []).append(r["doc_id"])
        positives.setdefault(r["query"], [])

print(f"{n_rows:,} pairs loaded")
print(f"  queries            {len(positives):,}")
print(f"  document pool      {len(doc_text):,} unique abstracts")
print(f"  positives          {sum(len(v) for v in positives.values()):,}")
print(f"  hard negatives     {sum(len(v) for v in hard.values()):,}")

# The pool has to be big enough that a random draw is meaningfully unrelated.
assert len(doc_text) > 5000, (
    f"only {len(doc_text)} documents recovered - the random negatives would be "
    "drawn from too narrow a pool to teach irrelevance"
)

## 4 · **Rebuild the negatives: 3 hard : 1 random** ← the entire point of this run

Per query with *P* positives, notebook 02 emitted **4·P hard** negatives. This
emits **3·P hard + 1·P random**.

What is deliberately held constant, so the comparison isolates one variable:

| | notebook 02 | notebook 03 |
|---|---|---|
| queries | 3,000 | 3,000 |
| positives | 15,079 | 15,079 |
| negatives | 60,316 | **60,316** |
| positive : negative | 1 : 4 | **1 : 4** |
| negative composition | **4 hard** | **3 hard + 1 random** |

**Which hard negatives get dropped:** the *last* one per positive. The file
preserves BM25 rank order, so the retained three are the highest-ranked — the
hardest, most confusable documents. The quarter that leaves is the weakest of
the hard negatives, which is the right quarter to spend.

**Three invariants, asserted rather than trusted:**

1. a random negative is **never** a true positive for its own query;
2. a random negative is **never** already a hard negative for that query (that
   would silently re-weight one document, not add a new signal);
3. the totals still come out at exactly 1 : 4.

In [ ]:
rng = random.Random(MIX_SEED)
all_ids = sorted(doc_text)          # sorted -> deterministic given MIX_SEED

mixed = []                          # the new training pairs
n_hard_kept = n_random = n_dropped = 0
retries_exhausted = 0

for query, pos_ids in positives.items():
    n_pos = len(pos_ids)
    if n_pos == 0:
        continue
    banned = set(pos_ids)

    # --- keep the HARDEST 3 per positive (file order == BM25 rank order) ---
    hard_ids = hard.get(query, [])
    keep_n = n_pos * HARD_PER_POSITIVE
    kept_hard = hard_ids[:keep_n]
    n_dropped += len(hard_ids) - len(kept_hard)
    n_hard_kept += len(kept_hard)
    banned.update(kept_hard)

    # --- add 1 RANDOM per positive, from the whole corpus ---
    want_random = n_pos * RANDOM_PER_POSITIVE
    drawn = []
    attempts = 0
    while len(drawn) < want_random and attempts < want_random * 50:
        attempts += 1
        cand = all_ids[rng.randrange(len(all_ids))]
        if cand in banned:            # invariants 1 and 2
            continue
        banned.add(cand)
        drawn.append(cand)
    if len(drawn) < want_random:
        retries_exhausted += 1
    n_random += len(drawn)

    for doc_id in pos_ids:
        mixed.append({"query": query, "text": doc_text[doc_id], "label": 1,
                      "doc_id": doc_id, "neg_kind": None})
    for doc_id in kept_hard:
        mixed.append({"query": query, "text": doc_text[doc_id], "label": 0,
                      "doc_id": doc_id, "neg_kind": "hard"})
    for doc_id in drawn:
        mixed.append({"query": query, "text": doc_text[doc_id], "label": 0,
                      "doc_id": doc_id, "neg_kind": "random"})

lab = Counter(p["label"] for p in mixed)
kind = Counter(p["neg_kind"] for p in mixed if p["label"] == 0)
total_neg = lab[0]

print(f"  pairs            {len(mixed):8,d}")
print(f"    positives      {lab[1]:8,d}")
print(f"    negatives      {lab[0]:8,d}   (ratio 1:{lab[0]/max(lab[1],1):.1f})")
print(f"      hard         {kind['hard']:8,d}   ({kind['hard']/max(total_neg,1):.1%} of negatives)")
print(f"      random       {kind['random']:8,d}   ({kind['random']/max(total_neg,1):.1%} of negatives)")
print(f"  hard negatives dropped (weakest per query): {n_dropped:,}")
if retries_exhausted:
    print(f"  !! {retries_exhausted} queries could not draw a full random quota")

# --- invariants -----------------------------------------------------------
for p in mixed:
    if p["neg_kind"] == "random":
        assert p["doc_id"] not in set(positives[p["query"]]), (
            "a random negative is a true positive for its own query")
expected_hard_share = HARD_PER_POSITIVE / (HARD_PER_POSITIVE + RANDOM_PER_POSITIVE)
actual_hard_share = kind["hard"] / max(total_neg, 1)
assert abs(actual_hard_share - expected_hard_share) < 0.02, (
    f"negative composition is {actual_hard_share:.1%} hard, expected "
    f"{expected_hard_share:.0%} - the mix did not come out as configured")
assert abs(lab[0] / max(lab[1], 1) - 4.0) < 0.05, "positive:negative ratio drifted from 1:4"
print("\n  OK: 1:4 ratio held, composition is "
      f"{actual_hard_share:.0%} hard / {1-actual_hard_share:.0%} random, no positive leaked in")

# --- eyeball it: T-331 asks for a spot-check, and so does this -------------
print("\n  spot-check - one query, its hardest negative and its random one:")
_q = next(q for q in positives if positives[q])
_h = next((p for p in mixed if p["query"] == _q and p["neg_kind"] == "hard"), None)
_r = next((p for p in mixed if p["query"] == _q and p["neg_kind"] == "random"), None)
print(f"    query   {_q[:100]}")
if _h: print(f"    hard    {_h['text'][:100]}...")
if _r: print(f"    random  {_r['text'][:100]}...")
print("    ^ the hard one should look topically adjacent; the random one should not.")

## 5 · Split **by survey**, not by row

⚠️ **The single thing most likely to silently invalidate this result.** One
survey contributes many chapter rows, so a row-level split puts chapters of the
*same survey* on both sides and inflates the dev score for free. Measured in
Phase 3B: 500 rows spanned only **63 distinct surveys**.

`MIX_SEED` deliberately did not touch this: the split is driven by `SEED`, so the
held-out surveys are **the same ones notebook 02 held out** and the two runs'
in-domain numbers are directly comparable.

In [ ]:
def survey_of(query):
    # The query is "<survey title> <chapter heading>"; chapters of one survey
    # share the title prefix. Grouping on a generous prefix keeps them together
    # without needing the raw field.
    return query[:80]

by_query = {}
for r in mixed:
    by_query.setdefault(r["query"], []).append(r)

surveys = sorted({survey_of(q) for q in by_query})
rng_split = random.Random(SEED)          # SEED, not MIX_SEED -- same split as nb 02
rng_split.shuffle(surveys)
n_dev = max(1, int(round(len(surveys) * DEV_FRAC)))
dev_surveys = set(surveys[:n_dev])

train_rows, dev_rows = [], []
for query, group in by_query.items():
    (dev_rows if survey_of(query) in dev_surveys else train_rows).extend(group)

assert not ({survey_of(r["query"]) for r in train_rows} &
            {survey_of(r["query"]) for r in dev_rows}), "survey leaked across the split"

for name, rs in [("train", train_rows), ("dev", dev_rows)]:
    c = Counter(r["label"] for r in rs)
    k = Counter(r["neg_kind"] for r in rs if r["label"] == 0)
    print(f"  {name:6s} {len(rs):7,d} pairs   pos {c[1]:6,d}  neg {c[0]:7,d} "
          f"(hard {k['hard']:6,d} / random {k['random']:5,d})   "
          f"queries {len({r['query'] for r in rs}):5,d}")

print(f"\n  OK: {len(surveys)} survey groups, {n_dev} held out, none on both sides")

## 6 · Build the CrossEncoder

`num_labels=1` with `BCEWithLogitsLoss` — the binary relevance objective from
`TRAINING_SPEC §2.3`. Same architecture the pipeline already runs at T-210, so a
successful result here is a **weight swap** and nothing downstream changes.

In [ ]:
from sentence_transformers import CrossEncoder
import torch, numpy as np, random as _random

torch.manual_seed(SEED); np.random.seed(SEED); _random.seed(SEED)

model = CrossEncoder(BASE_MODEL, num_labels=1, max_length=MAX_LENGTH)
print("loaded:", BASE_MODEL)
print("params:", sum(p.numel() for p in model.model.parameters()) / 1e6, "M")

## 7 · Baselines **before** training — ranking *and* the relevance control

Two measurements, both taken through the identical code path used after
training, so section 9 compares like with like rather than a remembered number.

**7a — ranking.** nDCG@10 is the metric to watch, not accuracy: this is a
ranking task and a model can classify pairs well while ordering badly.

**7b — the relevance control.** ★ *This is the diagnostic that caught the model
notebook 02 produced, and it caught it only after that model had already been
adopted.* It scores four deliberately off-topic documents — wind-turbine fatigue,
soil nitrogen, opera libretti, beta-lactamase crystallography — against each dev
query, and asks whether they beat the **median** real candidate. Median rather
than max, deliberately: a pool contains plenty of weak candidates, and beating
the worst of them is not a failure. Beating the typical one is.

The four control texts are **copied verbatim** from
`researchgpt.rerank.evaluate.OFF_TOPIC_CONTROLS`, so this number is comparable
with the local T-335 gate.

> **On the printed absolute scores.** The *violation rate* is invariant to any
> monotonic activation, so it is comparable across runs no matter what
> `sentence-transformers` does to the logits. The raw score column is only
> comparable within a single activation — it is there because *"stock says −11,
> tuned says +4"* is the memorable form of this failure.

In [ ]:
import numpy as np, torch

# Verbatim from researchgpt.rerank.evaluate.OFF_TOPIC_CONTROLS, so the number
# here and the number the local T-335 gate prints mean the same thing.
# The " * 3" is not decoration: a two-line document is out of distribution for a
# model trained on abstracts, and would be scored low for the wrong reason.
OFF_TOPIC_CONTROLS = [
    "Wind turbine blade fatigue under cyclic aerodynamic loading. We model crack "
    "propagation in glass-fibre composite spars and validate the model against "
    "full-scale accelerated fatigue tests on a 45-metre blade. " * 3,
    "Soil nitrogen mineralisation rates in temperate grassland under varying "
    "grazing intensity and rainfall. Field plots were sampled over three seasons "
    "and analysed for ammonium and nitrate concentration. " * 3,
    "A stylistic analysis of nineteenth-century Italian opera libretti and their "
    "metrical conventions, with particular attention to the versi sciolti of the "
    "Verdi middle period. " * 3,
    "Crystallographic determination of the beta-lactamase active site at 1.8 "
    "angstrom resolution, with implications for inhibitor design in resistant "
    "gram-negative bacteria. " * 3,
]


def raw_scores(m, pairs, batch_size=128):
    """Prefer un-activated logits, so the numbers match the local harness.

    sentence-transformers applies a sigmoid by default when num_labels==1, and
    has renamed this kwarg across versions. Fall through to the default rather
    than fail -- the violation rate does not depend on it.
    """
    for kw in ("activation_fct", "activation_fn"):
        try:
            return np.asarray(m.predict(pairs, batch_size=batch_size,
                                        show_progress_bar=False,
                                        **{kw: torch.nn.Identity()}), dtype=float)
        except TypeError:
            continue
    return np.asarray(m.predict(pairs, batch_size=batch_size,
                                show_progress_bar=False), dtype=float)


def group_by_query(rs):
    g = {}
    for r in rs:
        g.setdefault(r["query"], []).append(r)
    return g


def ndcg_at_k(labels_in_rank_order, k=10):
    gains = labels_in_rank_order[:k]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sorted(labels_in_rank_order, reverse=True)[:k]
    idcg = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_ranking(m, rs, name, k=10):
    groups = group_by_query(rs)
    ndcgs, recalls, precisions = [], [], []
    for query, items in groups.items():
        scores = raw_scores(m, [(query, it["text"]) for it in items])
        order = np.argsort(-scores)
        ranked = [items[i]["label"] for i in order]
        n_pos = sum(ranked)
        if n_pos == 0:
            continue
        ndcgs.append(ndcg_at_k(ranked, k))
        recalls.append(sum(ranked[:20]) / n_pos)
        precisions.append(sum(ranked[:12]) / 12)
    out = {"nDCG@10": float(np.mean(ndcgs)), "Recall@20": float(np.mean(recalls)),
           "Precision@12": float(np.mean(precisions)), "queries": len(ndcgs)}
    print(f"  {name:14s} nDCG@10 {out['nDCG@10']:.4f}   "
          f"Recall@20 {out['Recall@20']:.4f}   Precision@12 {out['Precision@12']:.4f}   "
          f"(n={out['queries']})")
    return out


def relevance_control(m, rs, name, max_queries=200):
    """Does the model still know what IRRELEVANT looks like?

    A violation is: the BEST off-topic control outscores the MEDIAN real
    candidate for that query. Notebook 02's model violated on 11 of 15 live
    queries; stock violated on 0 of 15.
    """
    groups = group_by_query(rs)
    queries = list(groups)[:max_queries]
    violations, margins, ctrl_scores, med_scores = 0, [], [], []
    for query in queries:
        items = groups[query]
        pool = np.sort(raw_scores(m, [(query, it["text"]) for it in items]))
        if len(pool) == 0:
            continue
        median = float(pool[len(pool) // 2])
        worst_control = float(np.max(raw_scores(m, [(query, c) for c in OFF_TOPIC_CONTROLS])))
        margins.append(median - worst_control)
        ctrl_scores.append(worst_control)
        med_scores.append(median)
        if worst_control >= median:
            violations += 1
    n = max(len(margins), 1)
    out = {"queries": len(margins), "violations": violations,
           "violation_rate": violations / n, "mean_margin": float(np.mean(margins)),
           "mean_control_score": float(np.mean(ctrl_scores)),
           "mean_pool_median": float(np.mean(med_scores))}
    print(f"  {name:14s} violations {out['violations']:3d}/{out['queries']:<3d} "
          f"({out['violation_rate']:6.1%})   margin {out['mean_margin']:+7.2f}   "
          f"off-topic score {out['mean_control_score']:+7.2f}   "
          f"pool median {out['mean_pool_median']:+7.2f}")
    return out


print("7a - ranking on the held-out dev surveys:")
stock_metrics = evaluate_ranking(model, dev_rows, "stock")

print("\n7b - relevance control (can it still recognise irrelevance?):")
stock_control = relevance_control(model, dev_rows, "stock")

## 8 · Train

Binary cross-entropy, 2 epochs, lr 2e-5, batch 32, 10% warmup — `TRAINING_SPEC
§2.3` exactly, and identical to notebook 02. **Expect ~20–30 minutes on a T4.**

In [ ]:
import math, time
from torch.utils.data import DataLoader
from sentence_transformers import InputExample

train_examples = [InputExample(texts=[r["query"], r["text"]], label=float(r["label"]))
                  for r in train_rows]
train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

steps_per_epoch = math.ceil(len(train_examples) / BATCH_SIZE)
warmup_steps = int(steps_per_epoch * EPOCHS * WARMUP_RATIO)
print(f"{len(train_examples):,} examples | {steps_per_epoch:,} steps/epoch | "
      f"{warmup_steps:,} warmup steps")

t0 = time.time()
model.fit(
    train_dataloader=train_loader,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LEARNING_RATE},
    weight_decay=WEIGHT_DECAY,
    use_amp=torch.cuda.is_available(),
    show_progress_bar=True,
)
train_seconds = time.time() - t0
print(f"\ntrained in {train_seconds/60:.1f} min")

## 9 · Result — ranking **and** relevance, stock vs fine-tuned

⚠️ **The ranking half is the IN-DOMAIN number and it is not the adoption gate.**
SciReviewGen is frozen at S2ORC 2020-07-05 and skews biomedical; your corpus is
CS/agentic-RAG. A large gain here with no gain on your own bibliography means you
fitted the benchmark, not the task (`TRAINING_SPEC §2.4`). **T-335 decides,
locally, on your own 25-paper bibliography, after int8 quantization.**

The relevance half is the one this whole run exists for. For reference, what the
**previous** (4-hard-negative) model did:

| | stock | notebook 02's model |
|---|---|---|
| live control violations | **0 / 15** | **11 / 15** |
| off-topic absolute score | ≈ −11 | **+3 … +5** |

In [ ]:
print("9a - ranking on the held-out dev surveys:")
tuned_metrics = evaluate_ranking(model, dev_rows, "fine-tuned")

print("\n9b - relevance control:")
tuned_control = relevance_control(model, dev_rows, "fine-tuned")

print("\n" + "=" * 78)
print("T-332R  IN-DOMAIN ranking (SciReviewGen dev, survey-disjoint)")
print("=" * 78)
print(f"  {'arm':14s} {'nDCG@10':>10s} {'Recall@20':>11s} {'Precision@12':>13s}")
for name, m in (("stock", stock_metrics), ("fine-tuned", tuned_metrics)):
    print(f"  {name:14s} {m['nDCG@10']:10.4f} {m['Recall@20']:11.4f} "
          f"{m['Precision@12']:13.4f}")
d = {k: tuned_metrics[k] - stock_metrics[k]
     for k in ("nDCG@10", "Recall@20", "Precision@12")}
print(f"  {'delta':14s} {d['nDCG@10']:+10.4f} {d['Recall@20']:+11.4f} "
      f"{d['Precision@12']:+13.4f}")

print("\n" + "=" * 78)
print("RELEVANCE CONTROL  - the reason this retrain exists")
print("=" * 78)
print(f"  {'arm':14s} {'violations':>12s} {'rate':>8s} {'margin':>9s} {'off-topic':>11s}")
for name, c in (("stock", stock_control), ("fine-tuned", tuned_control)):
    print(f"  {name:14s} {c['violations']:6d}/{c['queries']:<5d} "
          f"{c['violation_rate']:7.1%} {c['mean_margin']:+9.2f} "
          f"{c['mean_control_score']:+11.2f}")
print("=" * 78)

if d["nDCG@10"] <= 0.001:
    print("\n~0 in-domain gain. With only 3 hard negatives per positive the task is")
    print("easier than notebook 02's - if the gain has collapsed rather than merely")
    print("softened, the random share is too high. Try HARD_PER_POSITIVE=5,")
    print("RANDOM_PER_POSITIVE=1 before abandoning the approach.")

## 10 · Go / no-go — should this model even be downloaded?

Two conditions, both fixed in section 1 **before** the run:

1. **It must still rank better** — `nDCG@10` gain ≥ 0.001. A model that lost the
   discriminative signal along with the miscalibration is not progress.
2. **It must still recognise irrelevance** — control violation rate ≤ 20%.
   Notebook 02's model was at ~73%.

This is a **pre-filter, not the adoption decision.** T-335 still decides, locally,
out-of-domain, after int8 quantization. All this gate does is stop you spending a
download and a local eval cycle on a model that already failed the property it was
retrained to fix.

In [ ]:
rank_ok    = d["nDCG@10"] >= MIN_NDCG_GAIN
control_ok = tuned_control["violation_rate"] <= MAX_CONTROL_VIOLATION_RATE
regressed  = tuned_control["violation_rate"] > stock_control["violation_rate"] + 0.05

print("=" * 78)
print(f"  ranking gain        nDCG@10 {d['nDCG@10']:+.4f}  "
      f"(need >= {MIN_NDCG_GAIN:+.4f})   {'PASS' if rank_ok else 'FAIL'}")
print(f"  relevance retained  violations {tuned_control['violation_rate']:.1%}  "
      f"(need <= {MAX_CONTROL_VIOLATION_RATE:.0%})   {'PASS' if control_ok else 'FAIL'}")
print(f"  vs stock            stock is at {stock_control['violation_rate']:.1%}"
      f"{'   <-- WORSE THAN STOCK' if regressed else ''}")
print("=" * 78)

if rank_ok and control_ok:
    verdict = "PROCEED"
    print("  VERDICT: PROCEED to the local gate.")
    print("  The mixed negatives did what they were meant to: the model ranks better")
    print("  AND still knows what irrelevance looks like. Download both files below")
    print("  and run T-333 / T-334 / T-335 locally. T-335 is still the decision.")
elif rank_ok and not control_ok:
    verdict = "STOP - relevance prior still broken"
    print("  VERDICT: STOP. This is notebook 02's failure again.")
    print("  It ranks better and still cannot recognise irrelevance, so it would be")
    print("  rejected by T-335 exactly as its predecessor was. Raise the random share")
    print("  (RANDOM_PER_POSITIVE=2, i.e. 3:2) and re-run before spending a download.")
elif control_ok and not rank_ok:
    verdict = "STOP - no ranking gain"
    print("  VERDICT: STOP. Calibration was restored at the cost of the gain.")
    print("  Too many random negatives made the task trivial - the model learned topic")
    print("  matching, which BM25 already does. Go the other way: HARD_PER_POSITIVE=5.")
else:
    verdict = "STOP - failed both"
    print("  VERDICT: STOP. Neither condition held. Suspect the training config or the")
    print("  split before the negatives; check section 5 printed sane counts.")
    print("  TASKLIST's Aug 30 kill switch applies: keep the STOCK reranker, report the")
    print("  negative result, and move on. The pipeline is unaffected - it was always a")
    print("  weight swap - and the verifier satisfies the training requirement alone.")
print("=" * 78)

## 11 · Save

Download **both** files. The checkpoint feeds T-333 (ONNX + int8) and T-334/T-335
(the adoption gate); the metrics JSON supplies the retrain row of the
`TRAINING_SPEC §3` two-model table — including, this time, the relevance-control
numbers, which are the honest reporting line for T-705b whichever way the gate goes.

In [ ]:
import os, shutil, json

model.save(OUTPUT_DIR)

metrics = {
    "task": "T-332R",
    "note": "Retrain of T-332 with MIXED negatives after the T-335 revert.",
    "base_model": BASE_MODEL,
    "seed": SEED,
    "mix_seed": MIX_SEED,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "dev_frac": DEV_FRAC,
    "negatives": {
        "hard_per_positive": HARD_PER_POSITIVE,
        "random_per_positive": RANDOM_PER_POSITIVE,
        "hard_share_of_negatives": actual_hard_share,
        "hard_negatives_dropped": n_dropped,
        "previous_run": "4 hard : 0 random (notebook 02)",
    },
    "counts": {"train_pairs": len(train_rows), "dev_pairs": len(dev_rows),
               "survey_groups": len(surveys), "dev_surveys": n_dev,
               "document_pool": len(doc_text)},
    "in_domain": {"stock": stock_metrics, "fine_tuned": tuned_metrics},
    "relevance_control": {"stock": stock_control, "fine_tuned": tuned_control,
                          "off_topic_controls": len(OFF_TOPIC_CONTROLS)},
    "gate": {"min_ndcg_gain": MIN_NDCG_GAIN,
             "max_control_violation_rate": MAX_CONTROL_VIOLATION_RATE,
             "ranking_pass": bool(rank_ok), "control_pass": bool(control_ok),
             "verdict": verdict},
    "train_seconds": train_seconds,
    "gate_note": ("This gate is a PRE-FILTER. The adoption decision is T-335, run "
                  "locally against the project's own 25-paper bibliography, after "
                  "int8 quantization."),
}
with open("/kaggle/working/reranker_metrics_mixed.json", "w") as fh:
    json.dump(metrics, fh, indent=2)

shutil.make_archive("/kaggle/working/reranker-mixed-negatives", "zip", OUTPUT_DIR)
print("saved:")
for f in sorted(os.listdir("/kaggle/working")):
    p = os.path.join("/kaggle/working", f)
    if os.path.isfile(p):
        print(f"  {f:44s} {os.path.getsize(p)/1e6:8.2f} MB")

print(f"\nVERDICT WAS: {verdict}")
print("\nDownload reranker-mixed-negatives.zip AND reranker_metrics_mixed.json.")
print("Then locally:")
print("  python -m researchgpt.rerank.export_onnx --checkpoint data/models/reranker-mixed")
print("  python -m researchgpt.rerank.evaluate    --tuned-onnx data/models/reranker-mixed-onnx")